# 📝 과제 02: 루나 팬 대화 챗봇의 회귀 게이트

## 만들 결과

과제 01의 루나 팬 대화 챗봇을 운영하던 중 "답변 아래 `notice-fm` 같은 근거 ID가 무엇인지 모르겠다"는 팬 의견이 들어왔습니다. 담당자가 답변 프롬프트를 고쳐, 근거 자리에 근거 ID 대신 팬이 알아보기 쉬운 **문서 제목**(예: 팬미팅 공지)을 적게 하는 변경 요청(pull request)을 올렸습니다. 고친 프롬프트를 개정 프롬프트(`v2`)라고 부릅니다.

이 변경을 받아들여도 되는지 회귀 게이트로 판정합니다.

- 현재 프롬프트(`v1`)를 실행 조건과 함께 세 번 평가하고, 승인한 run을 기준선으로 지정해 파일로 저장합니다.
- 교안 02의 네 가지 판정 규칙을 pytest 테스트로 완성하고, 규칙을 어긴 경우에만 그 테스트가 실패하는지 가짜 기준선·결과 파일 쌍으로 확인합니다.
- 개정 프롬프트(`v2`)를 평가해 회귀 게이트로 판정합니다.
- 이 게이트를 변경 요청마다 돌릴 GitHub Actions 워크플로를 완성합니다.

교안 02의 run 기록, 기준선, 판정 규칙, pytest, CI를 루나 팬 대화 챗봇에 적용합니다. 루나와 관련 자료는 모두 수업용 가상 설정입니다.

## 준비: 데이터와 앱

- **자료:** 과제 01과 같은 `data/과제/`: 루나 공식 안내 문서 14편, 골든셋(정답셋) 12건, 문서 벡터
- **프로젝트 폴더:** `luna_gate_ci/`(테스트 파일 `tests/test_luna_gate.py`, 워크플로 파일 `.github/workflows/luna-gate.yml`, 자가채점용 가짜 파일 `fixtures/`)
- **순서:** 세 번 평가 -> 기준선 지정·저장 -> 회귀 테스트 완성 -> 개정 프롬프트 판정 -> 워크플로 완성
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 회귀 테스트와 워크플로는 노트북이 아니라 `luna_gate_ci/` 안의 파일을 편집기로 열어 작성합니다.
- **판정:** 판사(채점하는 LLM)의 판정은 실행마다 조금씩 달라질 수 있습니다. 자가채점은 지표 값이 아니라 결과의 형식(키, 개수, 순서)을 확인합니다.

아래 셀들은 과제 01에서 만든 부품(문서 검색, 현재 프롬프트의 답변 함수 `answer_v1`, 인용 검사와 두 채점 함수)과 교안 02의 보고서·기준선 함수를 준비합니다.

In [ ]:
import os
import warnings
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# DeepEval 이 사용 기록을 외부로 보내지 않게 합니다. deepeval 을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
# 채점 중 나오는 ipywidgets 설치 권고 경고를 숨깁니다.
warnings.filterwarnings("ignore", message='install "ipywidgets"')
from deepeval.models import OpenAIModel

# material_dir 는 data 와 mlflow.db 가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data" / "과제"

load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: 챗봇 답변을 만듭니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 미리 만들어 둔 문서 벡터와 같은 모델·차원으로 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 짧은 질문이라 자동 길이 검사·분할을 끕니다.

# 판사 모델: 답변 모델과 다른 모델로 채점합니다. 판사를 바꾸면 기준선도 다시 만들어야 합니다.
# gpt-5.6-luna 는 temperature 를 지정할 수 없는 모델이라 지정하지 않습니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-5.6-luna")
judge_llm = OpenAIModel(model=eval_model_name)

# MLflow 자동 기록(autolog)을 끕니다. 켜져 있으면 호출마다 실행 기록(trace)에 단계가 자동으로 생깁니다.
# - mlflow.langchain.autolog: LangChain 체인·프롬프트·모델 호출을 단계마다 입력, 출력, 걸린 시간, 토큰 수와 함께 기록합니다.
# - mlflow.openai.autolog: OpenAI SDK 호출(DeepEval 판사의 호출 포함)을 보낸 메시지, 모델 이름, 응답, 토큰 수와 함께 기록합니다.
# 꺼 두지 않으면 mlflow.genai.evaluate 가 평가 중에 이 둘을 잠시 켭니다. MLflow 3.14 에서는 그 기록이 늦게 닫히면 평가가 실행 기록을 찾지 못해 오류로 멈출 때가 있습니다.
# 실행 기록은 @mlflow.trace 로 직접 남깁니다.
mlflow.langchain.autolog(disable=True)
mlflow.openai.autolog(disable=True)

mlflow.set_tracking_uri(f"sqlite:///{material_dir / 'mlflow.db'}")
experiment = mlflow.set_experiment("luna-gate")
print("답변 모델:", model_name, "/ 판사 모델:", eval_model_name, "/ 실험:", experiment.name)

In [ ]:
import json

import pandas as pd
from langchain_chroma import Chroma


def read_jsonl(path):
    """jsonl 파일(한 줄에 JSON 하나)을 목록으로 읽습니다."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


luna_docs = read_jsonl(data_dir / "docs.jsonl")
luna_golden = read_jsonl(data_dir / "golden.jsonl")
# 인용을 검사할 때 근거 ID로 문서 본문을 찾는 사전입니다.
luna_texts = {}
for doc in luna_docs:
    luna_texts[doc["doc_id"]] = doc["text"]

vector_store = Chroma(collection_name="luna_guides", embedding_function=embedding_model,
                      persist_directory=str(data_dir / "chroma"))
print("문서:", len(luna_docs), "/ 골든 사례:", len(luna_golden))
display(pd.DataFrame(luna_golden)[["case_id", "input", "expected_output", "context", "relevant_doc_ids", "smoke"]])

In [ ]:
def evidence_id(doc):
    """문서에 부여한 근거 ID를 반환합니다."""
    return doc.metadata["source_id"]


@mlflow.trace(span_type="RETRIEVER")
def search_documents(search_query, top_k=4):
    """질문과 가까운 문서 top_k 개를 Document 목록으로 반환합니다."""
    return vector_store.similarity_search(search_query, k=top_k)


def format_documents(docs):
    """검색한 문서를 [근거 ID] 제목과 본문 형식으로 이어 붙입니다."""
    parts = []
    for doc in docs:
        parts.append(f"[{evidence_id(doc)}] {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n".join(parts)


for doc in search_documents("팬클럽 연회비는 얼마예요?"):
    print(evidence_id(doc), "|", doc.metadata["title"])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class Citation(BaseModel):
    evidence_id: str = Field(description="근거 ID를 넣습니다. 제공된 ID를 철자 그대로 씁니다")
    quote: str = Field(description="그 근거 원문에서 글자 그대로 복사한 구절을 넣습니다. 고치거나 요약하지 않습니다")


class Segment(BaseModel):
    text: str = Field(description="답변의 한 부분(보통 문장 하나)을 넣습니다. segments 의 text 를 순서대로 이으면 사용자에게 보여 줄 답변이 됩니다")
    citations: list[Citation] = Field(description="이 부분을 뒷받침하는 인용 목록입니다. 문서에서 가져온 사실이 아니면(연결 문장, 확인할 수 없다는 안내 등) 빈 목록입니다")


class CitedAnswer(BaseModel):
    segments: list[Segment] = Field(description="답변을 순서대로 나눈 부분 목록입니다")


# 현재 서비스 중인 답변 프롬프트(v1)입니다.
SYSTEM_V1 = ("당신은 버추얼 아이돌 루나의 팬 대화 챗봇입니다. 팬에게 따뜻한 해요체로, 루나 공식 안내 발췌로만 답하세요. "
             "답변을 부분(segments)으로 나눠 쓰고, 안내에서 가져온 사실을 쓴 부분마다 근거 ID와, 그 문서에서 글자 그대로 복사한 구절을 인용하세요. "
             "구절을 바꿔 쓰거나 요약하지 마세요. "
             "날짜·시각·금액·횟수와 조건을 빠뜨리지 마세요. "
             "안내에 답이 없으면 '제공된 루나 공식 안내에서 확인할 수 없어요.'라는 부분 하나만 인용 없이 쓰세요.")
prompt_v1 = ChatPromptTemplate.from_messages([("system", SYSTEM_V1), ("human", "질문: {question}\n\n공식 안내 발췌:\n{context}")])
chain_v1 = prompt_v1 | llm.with_structured_output(CitedAnswer)


@mlflow.trace
def answer_v1(question):
    """현재 프롬프트(v1)로 답하고, 검색한 근거 ID 를 함께 돌려줍니다."""
    docs = search_documents(question)
    result = chain_v1.invoke({"question": question, "context": format_documents(docs)})
    output = result.model_dump()

    # 부분을 순서대로 이어 사용자에게 보여 줄 답변을 만듭니다.
    texts = []
    for segment in output["segments"]:
        texts.append(segment["text"])
    output["answer"] = " ".join(texts)

    output["searched_ids"] = []
    for doc in docs:
        output["searched_ids"].append(evidence_id(doc))
    return output

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer


def normalize(text):
    """뜻과 상관없는 표기 차이(가운뎃점 모양, 줄바꿈, 연속 공백)를 하나로 맞춥니다."""
    for mark in ["･", "∙", "・", "•"]:
        text = text.replace(mark, "·")
    return " ".join(text.split())


def check_citation(citation, source_texts, searched_ids):
    """인용 하나를 근거 ID, 인용 문구 순으로 검사해 결과 문자열을 돌려줍니다."""
    cited_id = citation["evidence_id"]
    if cited_id not in source_texts:
        return "없는 근거 ID"
    if cited_id not in searched_ids:
        return "검색하지 않은 문서"
    if normalize(citation["quote"]) not in normalize(source_texts[cited_id]):
        return "원문에 없는 문구"
    return "확인"


@scorer
def citation_valid(outputs):
    problems = []
    for segment in outputs["segments"]:
        for citation in segment["citations"]:
            status = check_citation(citation, luna_texts, outputs["searched_ids"])
            if status != "확인":
                problems.append(f"{status}: {citation['evidence_id']}")
    if len(problems) == 0:
        return Feedback(value=True, rationale="붙은 인용이 모두 근거 ID·문구 대조를 통과함")
    return Feedback(value=False, rationale="; ".join(problems))


# 정답 판정의 판단 단계입니다. 교안 01과 같은 DeepEval G-Eval 정답 판정입니다.
CORRECTNESS_STEPS = [
    "expected_output 에 적힌 사실이 actual_output 에 모두 들어 있는지 본다.",
    "숫자, 기간, 대상, 조건이 하나라도 다르거나 빠지면 틀린 것으로 본다.",
    "expected_output 이 자료에서 확인할 수 없다는 안내이면, actual_output 도 확인할 수 없다고 답해야 맞다.",
    "판정 이유(reason)는 한국어로 쓴다.",
]


@scorer
def correctness(inputs, outputs, expectations):
    # 평가는 여러 질문을 동시에 채점하므로, 채점 함수가 불릴 때마다 GEval 지표를 새로 만들어 점수가 섞이지 않게 합니다.
    metric = GEval(
        name="정답 판정",
        evaluation_steps=CORRECTNESS_STEPS,
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        model=judge_llm,
        threshold=0.7,
        async_mode=False,  # 지표 안에서 비동기로 호출하지 않습니다(동시 채점 중 이벤트 루프 오류를 막습니다).
    )
    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], expected_output=expectations["expected_output"]), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)


# 회귀 게이트가 통과·실패를 판정할 때 쓰는 채점 함수(gate_scorers)와 그 이름 목록(JUDGED_SCORERS)입니다.
# 보고서·기준선·테스트가 이 이름으로 지표를 찾으므로 @scorer 함수 이름과 같아야 합니다.
gate_scorers = [correctness, citation_valid]
JUDGED_SCORERS = ["correctness", "citation_valid"]

In [ ]:
import subprocess
import sys

# 교안 02와 같은 보고서·기준선 함수입니다(지표와 지표 흔들림을 계산합니다).


def pass_rates(cases, case_ids):
    """case_ids 사례 가운데 채점 함수마다 통과한 비율을 계산합니다. 전체를 셀 때는 list(cases), 빠른 점검(smoke) 사례만 셀 때는 그 ID 목록을 넘깁니다."""
    rates = {}
    for name in JUDGED_SCORERS:
        passed = 0
        for case_id in case_ids:
            if cases[case_id][name] is True:
                passed += 1
        rates[name] = round(passed / len(case_ids), 4)
    return rates


def make_report(result, split):
    """평가 결과를 case_id 별 통과 여부와 지표로 정리합니다."""
    cases = {}
    errors = 0
    for _, row in result.result_df.iterrows():
        case = {}
        for name in JUDGED_SCORERS:
            value = row[f"{name}/value"]
            # 채점 중 오류가 난 칸은 값이 비어 있습니다. 통과로 세지 않고 오류로 셉니다.
            if pd.isna(value):
                errors += 1
                case[name] = None
            else:
                # 판정 값은 True/False 입니다. numpy 불리언도 bool 로 맞춥니다(빈 값은 위에서 오류로 셉니다).
                case[name] = bool(value == True)
        cases[row["case_id/value"]] = case
    return {"split": split, "run_id": result.run_id, "errors": errors,
            "metrics": pass_rates(cases, list(cases)), "cases": cases}


def make_baseline(reports, smoke_ids, absolute_floor, params):
    """반복 평가 보고서로 기준선을 만듭니다. reports[0] 이 승인한 기준선 run 입니다."""
    champion = reports[0]
    all_ids = list(champion["cases"])
    # 반복한 모든 run 에서 모든 판정을 통과한 사례(늘 통과한 사례)만 사례별 회귀 규칙에 씁니다.
    stable_cases = []
    for case_id in all_ids:
        always_passed = True
        for report in reports:
            for name in JUDGED_SCORERS:
                if report["cases"][case_id][name] is not True:
                    always_passed = False
        if always_passed:
            stable_cases.append(case_id)
    splits = {}
    for split_name, case_ids in [("full", all_ids), ("smoke", smoke_ids)]:
        allowed_drop = {}
        noise = {}
        for name in JUDGED_SCORERS:
            values = []
            for report in reports:
                values.append(pass_rates(report["cases"], case_ids)[name])
            noise[name] = round(max(values) - min(values), 4)
            # 허용 하락폭 = 지표 흔들림(계산한 값) + 1 / 사례 수(사례 1건만큼의 여유, 팀이 정하는 값)
            allowed_drop[name] = round(noise[name] + 1 / len(case_ids), 4)
        splits[split_name] = {"case_ids": case_ids, "metrics": pass_rates(champion["cases"], case_ids),
                              "noise": noise, "allowed_drop": allowed_drop, "absolute_floor": absolute_floor}
    return {"run_id": champion["run_id"], "params": params, "stable_cases": stable_cases, "splits": splits}


def run_pytest(folder):
    """폴더의 tests 를 pytest 로 실행하고 결과를 출력합니다. 종료 코드를 돌려줍니다."""
    command = [sys.executable, "-m", "pytest", "-vv", "--tb=no", "--color=no", "-p", "no:deepeval", "tests"]
    completed = subprocess.run(command, cwd=folder, capture_output=True, text=True)
    print("종료 코드:", completed.returncode)
    print(completed.stdout)
    return completed.returncode


# 과제 프로젝트 폴더입니다. 테스트 파일과 워크플로 파일, 자가채점용 가짜 파일이 들어 있습니다.
gate_dir = Path("luna_gate_ci")

# 골든셋을 평가 입력 형식으로 바꿉니다(과제 01의 to_records 와 같은 규칙).
luna_records = []
for case in luna_golden:
    expectations = {"case_id": case["case_id"], "expected_output": case["expected_output"]}
    luna_records.append({"inputs": {"question": case["input"]}, "expectations": expectations})
print("평가 입력:", len(luna_records), "건 / 게이트 채점 함수:", JUDGED_SCORERS)

## 1. 현재 프롬프트를 실행 조건과 함께 세 번 평가하세요

같은 설정을 반복 평가해야 판정이 흔들리는 사례를 "늘 통과한 사례"에서 뺄 수 있으므로, 현재 프롬프트를 세 번 평가하고 run마다 실행 조건을 남깁니다.

**작성:** 함수 `evaluate_run(run_name, dataset, predict_fn, scorers, params)`(교안 02와 같은 모양), 보고서 목록 `v1_reports`

**구현할 처리**
1. `evaluate_run`: `with mlflow.start_run(run_name=run_name):` 안에서 `params`의 키·값을 모두 `log_param`으로 남기고, `mlflow.genai.evaluate(data=dataset, predict_fn=predict_fn, scorers=scorers)`로 평가한 뒤 `log_metric("case_count", 결과표 행 수)`를 남기고 평가 결과를 돌려줍니다.
2. run 이름 `luna-v1-a`, `luna-v1-b`, `luna-v1-c` 순서로 `evaluate_run(run_name, luna_records, answer_v1, gate_scorers, v1_params)`처럼 평가하고, 결과마다 `make_report(결과, "full")`로 만든 보고서를 `v1_reports`에 차례로 담습니다.

**반환값:** `v1_reports`(보고서 3개, 첫 번째가 기준선 run의 보고서)

In [ ]:
# [작성] 실행 조건을 run 에 남기고 같은 run 안에서 평가하는 함수를 만들고, 현재 프롬프트를 세 번 평가하세요.
def evaluate_run(run_name, dataset, predict_fn, scorers, params):
    """실행 조건을 run 에 기록하고, 같은 run 안에서 평가합니다."""
    # 1. run 을 열고 조건 기록 -> 평가 -> 사례 수 기록
    ...


# 현재 프롬프트의 실행 조건(제공)
v1_params = {"prompt_version": "v1", "app_model": model_name, "judge_model": eval_model_name, "top_k": 4}

# 2. 세 번 평가해 보고서를 차례로 담기
v1_reports = []
...

for report in v1_reports:
    print("지표:", report["metrics"])

In [ ]:
# [자가채점] run 세 개에 실행 조건과 지표가 남았는지, 보고서가 사례 12건씩인지 확인합니다.
assert isinstance(v1_reports, list) and len(v1_reports) == 3, "보고서 세 개를 luna-v1-a, b, c 순서로 담으세요."
v1_runs = mlflow.search_runs(experiment_names=["luna-gate"], filter_string="params.prompt_version = 'v1'")
for run_name, report in zip(["luna-v1-a", "luna-v1-b", "luna-v1-c"], v1_reports):
    # 이번에 만든 보고서의 run 이 실험에 있고 이름이 맞는지 봅니다(예전에 돌린 run 으로는 통과하지 않습니다).
    matched = v1_runs[v1_runs["run_id"] == report["run_id"]]
    assert len(matched) == 1, f"{run_name} 보고서의 run 이 luna-gate 실험에 없습니다. start_run(run_name=...) 안에서 평가했는지와 log_param 을 확인하세요."
    latest = matched.iloc[0]
    assert latest["tags.mlflow.runName"] == run_name, f"보고서 순서와 run 이름이 다릅니다. {run_name} 자리에 {latest['tags.mlflow.runName']} 가 있습니다."
    for key in ["prompt_version", "app_model", "judge_model", "top_k"]:
        assert pd.notna(latest[f"params.{key}"]), f"{run_name}: log_param 으로 {key} 를 남기세요."
    assert latest["metrics.case_count"] == len(luna_records), f"{run_name}: log_metric('case_count', 결과표 행 수)를 남기세요."
    assert pd.notna(latest["metrics.correctness/mean"]), f"{run_name}: run 을 연 상태에서 평가해야 지표가 그 run 에 남습니다."
for report in v1_reports:
    assert report["split"] == "full" and len(report["cases"]) == len(luna_records), "make_report(결과, 'full') 로 만든 보고서를 담으세요."
print("반복 평가 확인 완료: run 3개, 보고서 3개")

## 2. 기준선을 지정하고 파일로 저장하세요

CI가 작업을 실행하는 가상 컴퓨터(러너)에는 MLflow 기록이 없으므로, 승인한 run을 MLflow에서는 태그로 표시하고 판정에 쓸 숫자는 저장소 안 파일로 남깁니다.

**작성:** 기준선 `luna_baseline`, 파일 `luna_gate_ci/baselines/baseline.json`

**구현할 처리**
1. 골든셋에서 `smoke`가 참인 사례의 `case_id`를 골든셋 순서대로 `smoke_ids`에 모읍니다.
2. `make_baseline(v1_reports, smoke_ids, {"correctness": 0.8, "citation_valid": 0.8}, v1_params)`로 `luna_baseline`을 만듭니다.
3. `luna-gate` 실험에서 태그 `baseline_role`이 `champion`인 run을 모두 `retired`로 바꾼 뒤, 기준선 run(`luna_baseline["run_id"]`)에 `baseline_role=champion`을 붙입니다. 끝난 run에 붙이므로 `MlflowClient().set_tag(run_id, 키, 값)`을 씁니다.
4. `gate_dir / "baselines" / "baseline.json"`에 `json.dumps(luna_baseline, ensure_ascii=False, indent=2)`로 저장합니다(폴더가 없으면 만듭니다).

**반환값:** `luna_baseline`(기준선 dict), 저장한 파일

In [ ]:
# [작성] 기준선을 만들어 champion 태그를 옮기고 파일로 저장하세요.
from mlflow import MlflowClient

# 1. smoke 사례 ID
smoke_ids = []
...

# 2. 기준선 만들기
luna_baseline = ...

# 3. 지금 champion 인 run(이전 기준선)의 태그를 retired 로 덮어쓴 뒤, 이번 기준선 run 에 champion 을 붙입니다.
client = MlflowClient()
...

# 4. 기준선 파일 저장
baseline_path = gate_dir / "baselines" / "baseline.json"
...

print("늘 통과한 사례:", luna_baseline["stable_cases"])
print("full 기준선 값:", luna_baseline["splits"]["full"]["metrics"], "/ 허용 하락폭:", luna_baseline["splits"]["full"]["allowed_drop"])

In [ ]:
# [자가채점] champion 태그가 기준선 run 하나에만 있는지, 저장한 파일이 기준선과 같은지 확인합니다.
champions = mlflow.search_runs(experiment_names=["luna-gate"], filter_string="tags.baseline_role = 'champion'")
assert list(champions["run_id"]) == [luna_baseline["run_id"]], "champion 태그는 기준선 run 하나에만 남아야 합니다. 이전 champion 을 retired 로 바꿨는지 확인하세요."
assert luna_baseline["run_id"] == v1_reports[0]["run_id"], "기준선 run 은 첫 번째 보고서(luna-v1-a)입니다."
smoke_ids = luna_baseline["splits"]["smoke"]["case_ids"]
smoke_of = {}
for case in luna_golden:
    smoke_of[case["case_id"]] = case["smoke"]
assert len(smoke_ids) == 6 and smoke_ids == sorted(smoke_ids), "smoke_ids 는 smoke 가 참인 6건을 골든셋 순서대로 담습니다."
for case_id in smoke_ids:
    assert smoke_of[case_id] is True, f"{case_id} 는 smoke 사례가 아닙니다."
assert luna_baseline["splits"]["full"]["absolute_floor"] == {"correctness": 0.8, "citation_valid": 0.8}, "절대 하한은 두 채점 함수 모두 0.8 입니다."
saved = json.loads((gate_dir / "baselines" / "baseline.json").read_text(encoding="utf-8"))
assert saved == luna_baseline, "luna_gate_ci/baselines/baseline.json 에 luna_baseline 을 그대로 저장하세요."
print("기준선 확인 완료: 늘 통과한 사례", len(luna_baseline["stable_cases"]), "/", len(luna_golden))

## 3. 회귀 테스트 파일을 완성하세요

판정 규칙을 pytest 테스트로 옮겨 두면 CI가 같은 규칙으로 매번 판정합니다. `luna_gate_ci/tests/test_luna_gate.py`에는 결과·기준선 파일을 읽는 부분과 테스트 함수 다섯 개의 이름이 들어 있습니다. 편집기로 열어 작성합니다.

각 함수의 `pytest.fail(...)` 줄을 지우고 `assert`를 씁니다.

**작성:** 파일 `luna_gate_ci/tests/test_luna_gate.py`의 테스트 함수 다섯 개

**구현할 처리**
1. `test_same_cases`: 이번 결과의 사례 ID 목록(`report["cases"]`의 키)과 기준선의 같은 범위 사례 목록(`split["case_ids"]`)이 순서와 관계없이 같은지 봅니다.
2. `test_no_scoring_errors`: `report["errors"]`가 0인지 봅니다.
3. `test_metric_floor(name)`: 이번 지표가 절대 하한(`split["absolute_floor"][name]`) 이상인지 봅니다.
4. `test_metric_drop(name)`: 기준선 지표에서 이번 지표를 뺀 값을 `round(..., 4)`로 반올림한 하락폭이 허용 하락폭(`split["allowed_drop"][name]`) 이하인지 봅니다. 같으면 통과입니다.
5. `test_stable_case_still_passes(case_id)`: 그 사례가 `JUDGED_SCORERS`의 모든 채점 함수에서 `True`인지 봅니다.
6. `assert` 메시지에는 무엇이 얼마나 어긋났는지(지표 이름과 값, 사례 ID)를 적습니다. CI 로그에서 이 메시지로 실패 원인을 읽습니다.

아래 자가채점 셀은 `fixtures/`의 가짜 기준선·결과 파일 쌍 일곱 개로 테스트를 돌립니다. 쌍 이름은 그 쌍이 확인하는 상황입니다(예: `04_drop`은 허용 하락폭만 넘은 결과, `02_pass_boundary`는 하락폭이 허용 하락폭과 같고 지표가 절대 하한과 같은 결과).

In [ ]:
# [자가채점] 가짜 기준선·결과 파일 쌍마다 작성한 테스트를 실행해, 실패해야 할 테스트만 실패하는지 확인합니다.
import shutil
import tempfile
import xml.etree.ElementTree as ET

# 가짜 파일 쌍마다 실패해야 하는 테스트 이름(빈 목록은 모두 통과해야 함)
expected_failures = {
    "01_pass_flaky_case": [],
    "02_pass_boundary": [],
    "03_case_regression": [
        "test_stable_case_still_passes[c03]",
        "test_stable_case_still_passes[c04]"
    ],
    "04_drop": [
        "test_metric_drop[citation_valid]"
    ],
    "05_floor": [
        "test_metric_floor[correctness]"
    ],
    "06_missing_case": [
        "test_same_cases"
    ],
    "07_scoring_error": [
        "test_no_scoring_errors"
    ]
}

test_file = gate_dir / "tests" / "test_luna_gate.py"
with tempfile.TemporaryDirectory() as workspace:
    for fixture_name, expected in expected_failures.items():
        # 작성한 테스트 파일과 그 쌍의 기준선·결과를 임시 프로젝트 폴더에 놓고 실행합니다.
        project = Path(workspace) / fixture_name
        (project / "tests").mkdir(parents=True)
        (project / "baselines").mkdir()
        (project / "reports").mkdir()
        shutil.copy(test_file, project / "tests" / "test_luna_gate.py")
        shutil.copy(gate_dir / "fixtures" / fixture_name / "baseline.json", project / "baselines" / "baseline.json")
        shutil.copy(gate_dir / "fixtures" / fixture_name / "results.json", project / "reports" / "results.json")
        subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:deepeval", "--junitxml", "junit.xml", "tests"],
                       cwd=project, capture_output=True, text=True)
        failed = []
        for case in ET.parse(project / "junit.xml").iter("testcase"):
            if case.find("failure") is not None or case.find("error") is not None:
                failed.append(case.get("name"))
        assert sorted(failed) == expected, f"[{fixture_name}] 실패한 테스트 {sorted(failed)} 가 기대 {expected} 와 다릅니다. 쌍 이름이 가리키는 규칙과 경계(같으면 통과)를 확인하세요."
print("회귀 테스트 확인 완료: 가짜 파일 쌍", len(expected_failures), "개 통과")

## 4. 개정 프롬프트를 평가하고 회귀 게이트로 판정하세요

비교하려고 같은 설정으로 평가한 둘째 run을 먼저 판정합니다. 이 결과(`v1_reports[1]`)는 지표 흔들림 계산에 쓴 run이라 허용 하락폭과 사례별 회귀 테스트를 반드시 통과합니다. 판정 대상은 개정 프롬프트의 결과입니다.

아래 셀은 개정 프롬프트 `SYSTEM_V2`와 체인 `chain_v2`를 준비합니다.

In [ ]:
# 개정 프롬프트(v2): 현재 프롬프트(v1)에서 인용 지시만 바꿨습니다(근거 ID 자리에 문서 제목).
SYSTEM_V2 = ("당신은 버추얼 아이돌 루나의 팬 대화 챗봇입니다. 팬에게 따뜻한 해요체로, 루나 공식 안내 발췌로만 답하세요. "
             "답변을 부분(segments)으로 나눠 쓰고, 안내에서 가져온 사실을 쓴 부분마다 근거와, 그 문서에서 글자 그대로 복사한 구절을 인용하세요. "
             "근거에는 팬이 알아보기 쉽도록 근거 ID 대신 문서 제목(예: 팬미팅 공지)을 적으세요. 구절을 바꿔 쓰거나 요약하지 마세요. "
             "날짜·시각·금액·횟수와 조건을 빠뜨리지 마세요. "
             "안내에 답이 없으면 '제공된 루나 공식 안내에서 확인할 수 없어요.'라는 부분 하나만 인용 없이 쓰세요.")
prompt_v2 = ChatPromptTemplate.from_messages([("system", SYSTEM_V2), ("human", "질문: {question}\n\n공식 안내 발췌:\n{context}")])
chain_v2 = prompt_v2 | llm.with_structured_output(CitedAnswer)

**작성:** 함수 `answer_v2(question)`, 결과 `v2_result`, 보고서 `v2_report`, 종료 코드 `exit_v1`, `exit_v2`

**구현할 처리**
1. `answer_v2`: `answer_v1`과 같은 순서로 쓰되 `chain_v1` 대신 `chain_v2`를 부릅니다.
2. `v2_params`: `dict(v1_params)`로 복사한 뒤 `prompt_version`만 `"v2"`로 바꿉니다.
3. `evaluate_run("luna-v2", luna_records, answer_v2, gate_scorers, v2_params)`로 평가하고 `make_report(v2_result, "full")`로 `v2_report`를 만듭니다.
4. `gate_dir / "reports" / "results.json"`에 같은 설정으로 평가한 둘째 run의 보고서(`v1_reports[1]`)를 저장하고 `run_pytest(gate_dir)`의 종료 코드를 `exit_v1`에 담습니다.
5. 같은 파일에 `v2_report`를 덮어써 저장하고 `run_pytest(gate_dir)`의 종료 코드를 `exit_v2`에 담습니다.

**반환값:** `v2_report`, `exit_v1`, `exit_v2`

In [ ]:
# [작성] 개정 프롬프트로 답하는 함수를 만들어 평가하고, 두 결과를 차례로 회귀 테스트로 판정하세요.
# 1. 개정 프롬프트의 답변 함수
@mlflow.trace
def answer_v2(question):
    ...


# 2~3. 실행 조건을 복사해 v2 로 바꾸고 평가
v2_params = ...
v2_result = ...
v2_report = ...
print("개정 프롬프트 지표:", v2_report["metrics"])

# 4. 같은 설정으로 평가한 둘째 run(v1_reports[1])의 결과로 테스트
# pytest 는 결과 파일만 읽으므로, 판정할 보고서를 같은 파일에 차례로 덮어쓰고 실행합니다.
results_path = gate_dir / "reports" / "results.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
print("[같은 설정 v1 둘째 run]")
exit_v1 = ...

# 5. 개정 프롬프트의 결과로 테스트
print("[개정 프롬프트]")
exit_v2 = ...

In [ ]:
# [자가채점] 개정 프롬프트 run(luna-v2) 의 실행 조건과 마지막에 저장한 결과 파일, 종료 코드의 모양을 확인합니다. 통과 여부는 실행마다 달라질 수 있어 보지 않습니다.
import inspect

v2_runs = mlflow.search_runs(experiment_names=["luna-gate"], filter_string="params.prompt_version = 'v2'")
assert v2_result.run_id in list(v2_runs["run_id"]), "evaluate_run('luna-v2', luna_records, answer_v2, gate_scorers, v2_params) 로 평가하세요. prompt_version 은 'v2' 입니다."
# answer_v2 가 개정 프롬프트 체인을 부르는지 봅니다(chain_v1 을 그대로 쓰면 v1 을 다시 평가한 것이 됩니다).
used_names = inspect.unwrap(answer_v2).__code__.co_names
assert "chain_v2" in used_names and "chain_v1" not in used_names, "answer_v2 는 chain_v1 대신 chain_v2 를 불러야 합니다."
assert v1_params["prompt_version"] == "v1", "v1_params 가 바뀌었습니다. dict(v1_params) 로 복사한 뒤 바꾸세요."
assert v2_report["split"] == "full" and len(v2_report["cases"]) == len(luna_records), "make_report(v2_result, 'full') 로 보고서를 만드세요."
saved = json.loads((gate_dir / "reports" / "results.json").read_text(encoding="utf-8"))
assert saved == v2_report, "results.json 에는 마지막으로 v2_report 가 저장되어 있어야 합니다."
assert isinstance(exit_v1, int) and isinstance(exit_v2, int), "run_pytest 가 돌려준 종료 코드를 exit_v1, exit_v2 에 담으세요."
print("개정 프롬프트 판정 확인 완료: 종료 코드", exit_v1, "->", exit_v2)

In [ ]:
# 개정 프롬프트에서 인용 검증에 실패한 사례와 이유입니다(이유는 앞 80자만).
for _, row in v2_result.result_df.sort_values("case_id/value").iterrows():
    if row["citation_valid/value"] != True:
        print(row["case_id/value"], "|", str(row["citation_valid/rationale"])[:80])

## 5. 회귀 게이트를 돌릴 CI 워크플로를 완성하세요

교안 02의 워크플로처럼 변경 요청에서는 대표 6건, main 통합 때는 전체 12건을 평가합니다. 여기에 실무에서 자주 더하는 조건 두 가지를 직접 찾아 넣습니다. `luna_gate_ci/.github/workflows/luna-gate.yml`을 편집기로 열어 `TODO` 여덟 곳을 채웁니다.

- **초안(draft) 변경 요청은 건너뛰기**: 작업 중인 변경 요청은 초안으로 열어 두고 커밋을 자주 더합니다. 커밋마다 평가하면 비용만 나가므로, 초안일 때는 건너뛰고 "검토 준비 완료(Ready for review)"로 바꾸는 순간 평가합니다.
- **`full-eval` 라벨로 전체 평가 요청**: 프롬프트를 크게 바꾼 변경 요청처럼 대표 6건만으로는 불안할 때, 리뷰어가 라벨을 붙여 병합 전에 전체 12건을 평가하게 합니다.

**작성:** 파일 `luna_gate_ci/.github/workflows/luna-gate.yml`

**구현할 처리**
1. 변경 요청의 활동 유형(`types`)에 기본 세 가지(열기, 커밋 추가, 다시 열기)를 그대로 두고, 초안을 검토 준비 완료로 바꿀 때와 라벨을 붙일 때를 더합니다.
2. 변경 요청에서 `app.py`, `run_eval.py`, `data/` 아래, `tests/` 아래, `baselines/` 아래 파일이 바뀔 때만 실행합니다(폴더는 `"data/**"`처럼 씁니다).
3. main 브랜치에 통합(push)될 때도 2와 같은 파일이 바뀌면 실행합니다.
4. 초안 변경 요청이면 `evaluate` 작업을 건너뜁니다. main 통합은 항상 실행합니다.
5. 환경 변수 `OPENAI_API_KEY`에 같은 이름의 저장소 비밀 변수를 넘깁니다.
6. `full-eval` 라벨이 붙은 변경 요청이면 전체 12건(`--split full`)을 평가합니다.
7~8. "결과 요약"과 "평가 산출물 저장" 단계는 앞 단계가 실패해도 실행합니다.

**찾아볼 곳(GitHub Docs)**: 1번은 "Events that trigger workflows" 문서의 `pull_request` 활동 유형, 4·6번은 "Contexts reference"의 `github.event`(웹훅 내용)와 "Evaluate expressions in workflows and actions"의 `contains` 함수입니다. 웹훅 내용에서 초안 여부와 라벨 이름 목록이 어느 키에 있는지는 "Webhook events and payloads"의 `pull_request` 항목에서 확인합니다.

In [ ]:
# [자가채점] 워크플로 파일을 읽어 요구사항 여덟 가지를 확인합니다.
import yaml

workflow_text = (gate_dir / ".github" / "workflows" / "luna-gate.yml").read_text(encoding="utf-8")
# 주석 줄의 TODO 안내는 남겨도 됩니다. 값 자리에 남은 TODO 만 봅니다.
for line in workflow_text.splitlines():
    if not line.strip().startswith("#"):
        assert "TODO" not in line, f"아직 채우지 않은 자리가 있습니다: {line.strip()}"
# BaseLoader 는 모든 값을 문자열로 읽어, 파일에 적힌 글자 그대로 확인할 수 있습니다.
workflow = yaml.load(workflow_text, Loader=yaml.BaseLoader)
triggers = workflow["on"]
pr_types = triggers["pull_request"].get("types", [])
for needed in ["opened", "synchronize", "reopened", "ready_for_review", "labeled"]:
    assert needed in pr_types, f"요구사항 1: pull_request 의 types 에 {needed!r} 가 없습니다."
for trigger in ["pull_request", "push"]:
    paths = triggers[trigger].get("paths", [])
    for needed in ["app.py", "run_eval.py", "data/**", "tests/**", "baselines/**"]:
        assert needed in paths, f"요구사항 2~3: {trigger} 의 paths 에 {needed!r} 가 없습니다."
assert triggers["push"].get("branches") == ["main"], "요구사항 3: push 는 main 브랜치에서만 실행합니다."
job = workflow["jobs"]["evaluate"]
condition = job.get("if", "").replace(" ", "")
assert "pull_request.draft" in condition and "push" in condition, "요구사항 4: 초안이면 건너뛰고 main 통합(push)은 항상 실행하는 조건인지 확인하세요."
assert job["env"]["OPENAI_API_KEY"].replace(" ", "") == "${{secrets.OPENAI_API_KEY}}", "요구사항 5(저장소 비밀 변수 넘기기)를 확인하세요."
checked = 0
for step in job["steps"]:
    if step.get("name") == "평가 실행":
        script = step["run"].replace(" ", "")
        assert "labels" in script and "'full-eval'" in script, "요구사항 6: full-eval 라벨이 붙은 변경 요청을 고르는 조건을 확인하세요."
    if step.get("name") in ("결과 요약", "평가 산출물 저장"):
        checked += 1
        step_condition = step.get("if", "").replace(" ", "").replace("${{", "").replace("}}", "")
        assert step_condition == "always()", f"요구사항 7~8: '{step['name']}' 단계가 앞 단계 실패에도 실행되는지 확인하세요."
assert checked == 2, "'평가 실행', '결과 요약', '평가 산출물 저장' 단계 이름은 바꾸지 마세요."
print("워크플로 확인 완료")